# Day 1 — ENSOcast data plumbing

Open ERSSTv5 over the network (no manual download), crop the tropical Pacific, convert to anomalies, pull ONI, and align it to months.

**Open in Colab after you push this repo:**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day1_data.ipynb)

## 0. Install packages

In [1]:
# xarray   — open/slice NetCDF climate grids (SST maps) with named dims (time, lat, lon)
# netCDF4  — backend so xarray can actually read .nc / OPeNDAP files
# numpy    — array math once data is out of xarray (stacking samples later, etc.)
# pandas   — read/align the ONI text table as a time series
# matplotlib — quick sanity plots of anomaly maps
!pip install -q xarray netCDF4 numpy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.4 MB/s eta 0:00:00


## 1. (Optional) Mount Drive and save outputs there

Skip this cell if you only want files for this runtime. Mount if you want `pacific_anom.nc` to survive after Colab disconnects.

In [ ]:
from pathlib import Path

USE_DRIVE = True  # set False to keep everything in /content

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
else:
    DATA_DIR = Path("/content/ensocast/data")

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", DATA_DIR)

## 2. Open ERSSTv5 monthly SST (streamed, no browser download)

This is the **Mean** monthly product — a stack of global SST maps, one per month since 1854.

In [ ]:
import xarray as xr

url = "https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.ersst.v5/sst.mnmean.nc"
ds = xr.open_dataset(url)
ds

## 3. Crop tropical Pacific, keep 1950 onward

In [ ]:
# lon is 0–360: 120E to 280E covers the tropical Pacific box used here
sst = ds["sst"].sel(lat=slice(30, -30), lon=slice(120, 280))
sst = sst.sel(time=slice("1950-01-01", None))
print(sst)

## 4. Convert to anomalies (departure from each month's long-term average)

In [ ]:
climatology = sst.groupby("time.month").mean("time")
anom = sst.groupby("time.month") - climatology
anom = anom.fillna(0.0)  # land / missing cells → 0

out_path = DATA_DIR / "pacific_anom.nc"
anom.to_netcdf(out_path)
print("Wrote", out_path)
print(anom)

## 5. Sanity plot — one anomaly map (should look like ocean patterns, not noise)

In [ ]:
import matplotlib.pyplot as plt

sample = anom.sel(time="1997-12", method="nearest")  # strong El Niño winter
sample.plot(cmap="RdBu_r", vmin=-3, vmax=3, figsize=(10, 4))
plt.title("Pacific SST anomaly (example: near Dec 1997)")
plt.show()

## 6. Pull ONI and map each season to its middle month

DJF → January, JFM → February, … so ONI lines up with the SST monthly time axis.

In [ ]:
import pandas as pd
import numpy as np

SEAS_TO_MONTH = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4,
    "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8,
    "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

oni = pd.read_csv(
    "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt",
    sep=r"\s+",
)
oni["month"] = oni["SEAS"].map(SEAS_TO_MONTH)
oni["time"] = pd.to_datetime(
    dict(year=oni["YR"], month=oni["month"], day=1)
)
oni = oni.set_index("time").sort_index()
oni = oni.loc["1950-01-01":]
oni[["SEAS", "YR", "ANOM"]].head(12)

## 7. Align ONI to every SST month

In [ ]:
sst_times = pd.to_datetime(anom["time"].values)
# normalize to month start so it matches our ONI index
sst_month_starts = sst_times.to_period("M").to_timestamp()

oni_aligned = oni["ANOM"].reindex(sst_month_starts)
missing = int(oni_aligned.isna().sum())
print(f"Months in SST stack: {len(sst_month_starts)}")
print(f"Missing ONI after align: {missing}")

# drop months without ONI (usually the very newest SST months)
valid = ~oni_aligned.isna()
anom_aligned = anom.isel(time=np.where(valid.to_numpy())[0])
oni_by_month = oni_aligned[valid].to_numpy(dtype="float32")

print("Aligned anomaly shape:", anom_aligned.shape)
print("ONI length:", len(oni_by_month))
print("Latest aligned month:", pd.Timestamp(anom_aligned.time.values[-1]).date())
print("Latest ONI:", float(oni_by_month[-1]))

In [ ]:
# persist aligned ONI for Day 2
oni_out = DATA_DIR / "oni_monthly.csv"
pd.DataFrame(
    {
        "time": pd.to_datetime(anom_aligned.time.values),
        "oni": oni_by_month,
    }
).to_csv(oni_out, index=False)
print("Wrote", oni_out)
print("Day 1 checkpoint: anomaly maps + monthly ONI ready.")